In [16]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

In [17]:
# hyperparameters

batch_size = 64     # how many independent sequences will we process in parallel
block_size = 256     # what is the maximum context length for predictions
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = "cuda" if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 256
n_head = 8
n_layer = 6
dropout = 0.3
min_lr = 3e-5

warmup_iters = 500
lr_decay_iters = 5000

torch.manual_seed(1337)

In [18]:
# read training data and map all the unique characters

with open('lotr_trilogy_movie_script.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [19]:
# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [20]:
def get_lr(iter, warmup_iters=500, lr_decay_iters=5000, min_lr=3e-5):
    
    # warmup
    if iter < warmup_iters:
        return learning_rate * (iter+1) / warmup_iters 
    
    # decay
    if iter > lr_decay_iters:
        return min_lr
    
    # Cosine decay
    decay_ratio = (iter - warmup_iters) / (lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + np.cos(np.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@ torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


class Head(nn.Module):
    # one head of self-attention

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * C**(-0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    # multiple heads of self-attention in parallel


    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.dropout(self.proj(out))
        return out
    

class FeedForward(nn.Module):
    # a single linear layer followed by a non-linearity

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),      # GELU activation
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    # transformer block: communication followed by computation

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size=head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
    
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
        self.apply(self._init_weights)  # better weight initialization
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B, T) tensor of integers
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :]
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


In [21]:
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()

model = BigramLanguageModel()
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M_parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    # Update learning rate
    lr = get_lr(iter, warmup_iters=warmup_iters, lr_decay_iters=lr_decay_iters, min_lr=min_lr)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    
    # Evaluation (OUTSIDE the param_group loop!)
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"Step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, lr {lr:.6f}")
    
    # Training step (OUTSIDE the param_group loop!)
    xb, yb = get_batch('train')
    
    with autocast():
        logits, loss = model(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    # logits, loss = model(xb, yb)
    # optimizer.zero_grad(set_to_none=True)
    # loss.backward()
    # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    # optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

4.843605 M_parameters
Step 0: train loss 4.4930, val loss 4.4812, lr 0.000001
Step 500: train loss 2.4719, val loss 2.4467, lr 0.000300
Step 1000: train loss 1.7760, val loss 1.7370, lr 0.000292
Step 1500: train loss 1.5065, val loss 1.5248, lr 0.000268
Step 2000: train loss 1.3696, val loss 1.4508, lr 0.000232
Step 2500: train loss 1.2530, val loss 1.3843, lr 0.000188
Step 3000: train loss 1.1707, val loss 1.3466, lr 0.000142
Step 3500: train loss 1.1082, val loss 1.3312, lr 0.000098
Step 4000: train loss 1.0615, val loss 1.3210, lr 0.000062
Step 4500: train loss 1.0302, val loss 1.3202, lr 0.000038
Step 4999: train loss 1.0093, val loss 1.3122, lr 0.000030

CLOSE ON: . . . WO ROCKLY SPAKES -
illy towards in the TORCHES! The WHITE WHITE is bursts of a
trun is through her LABNNKIGHT GREEALIN exting to the
GARMSS open.

DENETHOR
(waith willing my - the Gates! Saruman are
Rohan in my bready in the extar . . ..

AC ON: A TUDARK h PALATE -, shalding the ROHIAD .... he looks back
illars the

In [22]:
print(f"Dataset size: {len(text):,} characters ({len(text)/1000:.0f}K)")

Dataset size: 424,020 characters (424K)


In [23]:
test_iters = [0, 1, 10, 100, 500, 1000, 5000, 10000, 15000, 20000]
for i in test_iters:
    lr = get_lr(i, lr_decay_iters=lr_decay_iters,min_lr=min_lr)
    print(f"Iter {i:5d}: LR = {lr:.8f}")

Iter     0: LR = 0.00000060
Iter     1: LR = 0.00000120
Iter    10: LR = 0.00000660
Iter   100: LR = 0.00006060
Iter   500: LR = 0.00030000
Iter  1000: LR = 0.00029186
Iter  5000: LR = 0.00003000
Iter 10000: LR = 0.00003000
Iter 15000: LR = 0.00003000
Iter 20000: LR = 0.00003000
